# 1. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import requests
import glob
import pandas as pd

pd.set_option("display.max_columns", 500)
pd.options.plotting.backend = "plotly"

# 2. Data : Télécharger les fichiers sources depuis PRIM IDFM

In [ ]:
# Télécharger les fichiers sources depuis la Plateforme Régionale d'Information pour la Mobilité (PRIM) d'Ile-de-France Mobilités et enregistrer localement.

# Données de validation de titres par trimestre.

url_validations_t1 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-1er-trimestre/exports/csv"

url_validations_t2 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-2eme-trimestre/exports/csv"

url_validations_t3 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-3eme-trimestre/exports/csv"

url_validations_t4 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-4eme-trimestre/exports/csv"


# Chemin du répertoire pour mettre les fichiers csv de validation.
folder_path_validations_multiples = os.path.join(
    "..", "data", "raw", "validations_multiples"
)

# Vérifier et créer le répertoire de destination s'il n'existe pas.

if not os.path.exists(folder_path_validations_multiples):
    os.makedirs(folder_path_validations_multiples)
    print(f"Répertoire créé : {folder_path_validations_multiples}")


# Liste et emplacements des gares / stations.

url_stations = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/emplacement-des-gares-idf-data-generalisee/exports/csv"


# Chemin du répertoire pour mettre les fichiers csv de liste et emplacement des gares / stations.
folder_path_stations = os.path.join("..", "data", "raw", "stations")

# Vérifier et créer le répertoire de destination s'il n'existe pas.

if not os.path.exists(folder_path_stations):
    os.makedirs(folder_path_stations)
    print(f"Répertoire créé : {folder_path_stations}")


# Définir une fonction pour télécharger les sources.


def telecharger_csv(url, destination, fichier):
    # Chemin complet du fichier.
    filepath = os.path.join(destination, fichier)
    try:
        print(f"Téléchargement en cours depuis : {url}")

        # Effectuer une requête HTTP.
        reponse = requests.get(url, timeout=10)

        # Erreur si le statut HTTP n'est pas bon (404, 500).
        reponse.raise_for_status()

        # Ecrire le contenu dans un fichier (en mode binaire).
        with open(filepath, "wb") as f:
            f.write(reponse.content)

        print(f"Fichier enregistré sous : {filepath}")

    except requests.exceptions.RequestException as e:
        print(f"Erreur lors du téléchargement : {e}")


if __name__ == "__main__":
    # Télécharger les fichiers de validations.

    url = url_validations_t1
    destination = folder_path_validations_multiples
    fichier = "validations_t1.csv"

    telecharger_csv(url, destination, fichier)

    url = url_validations_t2
    destination = folder_path_validations_multiples
    fichier = "validations_t2.csv"

    telecharger_csv(url, destination, fichier)

    url = url_validations_t3
    destination = folder_path_validations_multiples
    fichier = "validations_t3.csv"

    telecharger_csv(url, destination, fichier)

    url = url_validations_t4
    destination = folder_path_validations_multiples
    fichier = "validations_t4.csv"

    telecharger_csv(url, destination, fichier)

    # Télécharger les fichiers de liste et emplacements des gares / stations.

    url = url_stations
    destination = folder_path_stations
    fichier = "stations.csv"

    telecharger_csv(url, destination, fichier)

# 3. EDA

## 3.1. Données de validation de titres.

In [ ]:
# Objectif : combiner les fichiers sources de validations en un seul fichier csv.

# Nom de colonne "ida" pour "validations_t1.csv" alors que les autres csv sont en "id_zdc"
filepath_t1 = os.path.join(
    "..", "data", "raw", "validations_multiples", "validations_t1.csv"
)
df_t1 = pd.read_csv(filepath_t1, sep=";")
df_t1

In [ ]:
# Correction nom de colonne "ida" en "id_zdc"
df_t1 = df_t1.rename(columns={"ida": "id_zdc"})
df_t1

In [ ]:
# Enregistrer en csv
filepath_validations_multiples = os.path.join(
    "..", "data", "raw", "validations_multiples", "validations_t1.csv"
)
df_t1.to_csv(filepath_validations_multiples, sep=";", index=False, encoding="utf-8-sig")

In [ ]:
# 4 fichiers csv de validations de titres (1 par trimestre) pour 2025 à combiner

# Chemin du répertoire où se trouvent les fichiers csv à combiner
folder_path_validations_multiples = os.path.join(
    "..", "data", "raw", "validations_multiples"
)

# Récuperer tous les fichiers csv dans ce répertoire
validations = glob.glob(os.path.join(folder_path_validations_multiples, "*.csv"))

# List comprehension pour lire et combiner / concaténer
validations_df = pd.concat(
    (pd.read_csv(f, sep=";") for f in validations), ignore_index=True
)

In [ ]:
# Chemin du répertoire pour mettre le fichiers csv de validation combiné.
folder_path_validations = os.path.join("..", "data", "raw", "validations")

# Vérifier et créer le répertoire de destination s'il n'existe pas.
if not os.path.exists(folder_path_validations):
    os.makedirs(folder_path_validations)
    print(f"Répertoire créé : {folder_path_validations}")

# Exporter vers un fichier csv.
combined_validations_filepath = os.path.join(
    "..", "data", "raw", "validations", "validations.csv"
)
validations_df.to_csv(combined_validations_filepath, index=False)

print(
    f"Les {len(validations)} fichiers csv sont combinés avec succès dans : {combined_validations_filepath} !"
)

### 3.1.1. Vue globale des données.

In [ ]:
validations_df.head()

Colonnes interessantes pour l'analyse : jour, id_zdc pour faire les jointures, categorie_titre, nb_vald.

In [ ]:
print(validations_df.shape)
print(
    f"Données de validation : Nombre de ligne : {validations_df.shape[0]}, Nombre de colonnes : {validations_df.shape[1]} "
)

In [ ]:
validations_df.columns

In [ ]:
validations_df.describe()

In [ ]:
validations_df.describe(include=object)

In [ ]:
validations_df.info()

In [ ]:
validations_df["categorie_titre"].value_counts()

Doublon catégorie pour "Solidarité" et "Solidarite"  
NON DEFINI désigne une anomalie chez idfm. Ne pas grouper avec "Autres titres".

### 3.1.2. Data cleaning.

In [ ]:
# Formater colonne "jour".
validations_df["jour"] = pd.to_datetime(validations_df["jour"])

In [ ]:
validations_df.info()

In [ ]:
# Ajout colonnes mois.
validations_df["mois"] = validations_df["jour"].dt.month

# Ajout colonnes jour de la semaine (0 pour Lundi et 6 pour Dimanche).
validations_df["jour_sem_num"] = validations_df["jour"].dt.weekday

In [ ]:
validations_df.head()

In [ ]:
# Correction "categorie_titre": renommer "Contrat Solidarité Transport" par "Contrat Solidarite Transport"
validations_df["categorie_titre"] = validations_df["categorie_titre"].replace(
    "Contrat Solidarité Transport", "Contrat Solidarite Transport"
)

In [ ]:
validations_df["categorie_titre"].value_counts()

In [ ]:
# Valeurs manquantes.
validations_df.isna().sum()

1622 valeurs manquantes sur 2 millions pour la colonne id_zdc

In [ ]:
# A quoi correspondent les valeurs manquantes ?
validations_df[validations_df["id_zdc"].isnull()]

In [ ]:
# Attribuer une "id_zdc" à "Aéroport d'Orly" (information depuis la table stations)
validations_df.loc[validations_df["libelle_arret"] == "Aéroport d'Orly", "id_zdc"] = (
    63284
)

In [ ]:
# Supprimer les lignes dont "libelle_arret" == "Inconnu" (séléctionner les lignes où le libellé est différent de "Inconnu" et écraser l'acien dataframe).
validations_df = validations_df[validations_df["libelle_arret"] != "Inconnu"]

In [ ]:
# Modifier id_zdc de float64 en int64.
validations_df["id_zdc"] = validations_df["id_zdc"].astype(int)

In [ ]:
# Modifier id_zdc de int64 en str.
validations_df["id_zdc"] = validations_df["id_zdc"].astype(str)

In [ ]:
# Correction "id_zdc" de validations pour correspondre aux "id_ref_zdc" des stations.

dictionnaire_id_zdc = {
    "59577": "478855",
    "62737": "478505",
    "63650": "463850",
    "67747": "462934",
    "69531": "463754",
    "71219": "473829",
    "71245": "71229",
    "71282": "479068",
    "71686": "71697",
    "71743": "463564",
    "72059": "478883",
    "72219": "72225",
    "73616": "478885",
    "73792": "478926",
    "73794": "474151",
    "73795": "474152",
    "74040": "71139",
    "74371": "463843",
    "93066": "490784",
    "412697": "479919",
    "425819": "66915",
    "474149": "73615",
    "474150": "71229",
    "482368": "73688",
    "492980": "71271",
}

validations_df["id_zdc"] = validations_df["id_zdc"].replace(dictionnaire_id_zdc)

In [ ]:
# Modifier id_zdc Magenta par 478733
validations_df.loc[
    (validations_df["id_zdc"] == "74000")
    & (validations_df["libelle_arret"] == "MAGENTA"),
    "id_zdc",
] = "478733"

In [ ]:
validations_df[validations_df["libelle_arret"] == "MAGENTA"]

In [ ]:
# Modifier id_zdc La Chapelle par 71434
validations_df.loc[
    (validations_df["id_zdc"] == "74000")
    & (validations_df["libelle_arret"] == "LA CHAPELLE"),
    "id_zdc",
] = "71434"

In [ ]:
validations_df[validations_df["libelle_arret"] == "LA CHAPELLE"]

In [ ]:
# Pas de doublons.
validations_df.duplicated().sum()

## 3.2. Liste et emplacement des gares / stations.

In [ ]:
# Liste des stations généralisée (une station peut regrouper plusieurs lignes de transport associées)
filepath_stations = os.path.join("..", "data", "raw", "stations", "stations.csv")
stations_df = pd.read_csv(filepath_stations, sep=";")

### 3.2.1. Vue globale des données.

In [ ]:
stations_df.head()

Colonnes interessantes pour l'analyse :  
geo_point_2d pour carte  
id_ref_zdc pour les jointures  
nom_zdc  
res_com  
mode  
train, rer, metro, tramway, val pour identifier les modes  
tertrain, terrer, termetro, tertram, terval pour voir l'impact des terminus  
exploitant  
principal 

In [ ]:
print(stations_df.shape)
print(
    f"Données de validation : Nombre de ligne : {stations_df.shape[0]}, Nombre de colonnes : {stations_df.shape[1]} "
)

In [ ]:
stations_df.columns

In [ ]:
stations_df.describe().T

In [ ]:
stations_df.describe(include=object).T

In [ ]:
stations_df.info()

### 3.2.2. Data cleaning.

In [ ]:
# Création Colonnes latitude et longitude.
stations_df[["latitude", "longitude"]] = (
    stations_df["geo_point_2d"].str.split(", ", expand=True).astype(float)
)

In [ ]:
stations_df.head()

In [ ]:
# Modifier id_zdc en str.
stations_df["id_ref_zdc"] = stations_df["id_ref_zdc"].astype(str)

In [ ]:
# Valeurs manquantes.
stations_df.isna().sum()

Pas de valeurs manquantes sauf pour nom_so_gar (nom sous gare) et nom_su_gar (nom sur gare) qui est normal : toutes les stations n'ont pas forcémment de sur/sous nom.

In [ ]:
# Remplacer dans "termetro": "METRO 14" par 1 (valeur 1 ou 0 uniquement).
stations_df.loc[stations_df["termetro"] == "METRO 14", "termetro"] = 1

In [ ]:
# Modifier termetro de object à int64.
stations_df["termetro"] = stations_df["termetro"].astype(int)

In [ ]:
# Correction de noms de stations et res_com.

stations_df.loc[stations_df["id_ref_zdc"] == "478733", "nom_zdc"] = "Magenta"
stations_df.loc[stations_df["id_ref_zdc"] == "478855", "nom_zdc"] = "Etampes"
stations_df.loc[stations_df["id_ref_zdc"] == "71697", "nom_zdc"] = "Avron"
stations_df.loc[stations_df["id_ref_zdc"] == "479928", "nom_zdc"] = "Buzenval"
stations_df.loc[stations_df["id_ref_zdc"] == "73688", "nom_zdc"] = "Havre-Caumartin"


stations_df.loc[stations_df["id_ref_zdc"] == "71697", "res_com"] = "METRO 9"
stations_df.loc[stations_df["id_ref_zdc"] == "479928", "res_com"] = "METRO 2"
stations_df.loc[stations_df["id_ref_zdc"] == "73688", "res_com"] = "METRO 3 / METRO 9"


# Correction Châtelet les Halles.

stations_df.loc[
    (stations_df["id_ref_zdc"] == "474151")
    & (stations_df["nom_zdc"] == "Châtelet les Halles"),
    "res_com",
] = "RER A / RER B / RER D / METRO 4"
stations_df.loc[
    (stations_df["id_ref_zdc"] == "474151")
    & (stations_df["nom_zdc"] == "Châtelet les Halles"),
    "mode",
] = "RER / METRO"
stations_df.loc[
    (stations_df["id_ref_zdc"] == "474151")
    & (stations_df["nom_zdc"] == "Châtelet les Halles"),
    "metro",
] = 1
stations_df["metro"] = stations_df["metro"].astype(int)

In [ ]:
stations_df[stations_df["id_ref_zdc"] == "474151"]

In [ ]:
# Modifier id_ref_zdc Château Landon par 73615
stations_df.loc[
    (stations_df["id_ref_zdc"] == "71359")
    & (stations_df["nom_zdc"] == "Château Landon"),
    "id_ref_zdc",
] = "73615"

In [ ]:
stations_df[
    (stations_df["id_ref_zdc"] == "73615")
    & (stations_df["nom_zdc"] == "Château Landon")
]

In [ ]:
# Suppression des doublons id_ref_zdc

stations_df = stations_df[
    ~(
        (stations_df["id_ref_zdc"] == "63284")
        & (stations_df["nom_zdc"] == "Aéroport Orly 1-2-3 (Terminal Ouest)")
    )
]
stations_df = stations_df[
    ~(
        (stations_df["id_ref_zdc"] == "69677")
        & (stations_df["nom_zdc"] == "Pont de Rungis Aéroport d'Orly")
    )
]
stations_df = stations_df[
    ~((stations_df["id_ref_zdc"] == "71229") & (stations_df["nom_zdc"] == "La Muette"))
]
stations_df = stations_df[
    ~(
        (stations_df["id_ref_zdc"] == "71607")
        & (stations_df["nom_zdc"] == "Gare de Bercy")
    )
]
stations_df = stations_df[
    ~(
        (stations_df["id_ref_zdc"] == "73620")
        & (stations_df["nom_zdc"] == "Cluny La Sorbonne")
    )
]
stations_df = stations_df[
    ~(
        (stations_df["id_ref_zdc"] == "73753")
        & (stations_df["nom_zdc"] == "Porte de Thiais (Marché international)")
    )
]
stations_df = stations_df[
    ~(
        (stations_df["id_ref_zdc"] == "474151")
        & (stations_df["nom_zdc"] == "Les Halles")
    )
]

In [ ]:
# Ajout colonne "nb_lignes" nombre de ligne selon contenu de "res_com".
stations_df["nb_lignes"] = stations_df["res_com"].str.count("/") + 1

In [ ]:
stations_df.head()

## 3.3. Merge

In [ ]:
# Merge
df = validations_df.merge(
    stations_df, how="left", left_on="id_zdc", right_on="id_ref_zdc"
)

In [ ]:
df.head()

In [ ]:
# Supprimer les colonnes non utiles.
colonnes_a_supprimer = [
    "code_stif_trns",
    "code_stif_res",
    "code_stif_arret",
    "nom_long",
    "nom_so_gar",
    "nom_su_gar",
    "geo_point_2d",
    "geo_shape",
    "objectid_1",
    "codeunique",
    "id_ref_zda",
    "nom_zda",
    "idrefliga",
    "idrefligc",
    "x",
    "y",
]

df = df.drop(columns=colonnes_a_supprimer)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
# Modifier "nb_vald" de float64 en int64.
df["nb_vald"] = df["nb_vald"].astype(int)

In [ ]:
# List comprehension pour calculer le nombre de validations par lignes de transport.

lignes = [
    "RER A",
    "RER B",
    "RER C",
    "RER D",
    "RER E",
    "METRO 1",
    "METRO 2",
    "METRO 3",
    "METRO 3bis",
    "METRO 4",
    "METRO 5",
    "METRO 6",
    "METRO 7",
    "METRO 7bis",
    "METRO 8",
    "METRO 9",
    "METRO 10",
    "METRO 11",
    "METRO 12",
    "METRO 13",
    "METRO 14",
    "TRAIN H",
    "TRAIN J",
    "TRAIN K",
    "TRAIN L",
    "TRAIN N",
    "TRAIN P",
    "TRAIN R",
    "TRAIN V",
    "TRAM 1",
    "TRAM 2",
    "TRAM 3",
    "TRAM 3a",
    "TRAM 3b",
    "TRAM 4",
    "TRAM 5",
    "TRAM 6",
    "TRAM 7",
    "TRAM 8",
    "TRAM 9",
    "TRAM 10",
    "TRAM 11",
    "TRAM 12",
    "TRAM 13",
    "TRAM 14",
    "CABLE 1",
    "CDGVAL",
    "FUNICULAIRE MONTMARTRE",
]

lignes_df = pd.DataFrame(
    {
        "Ligne": lignes,
        "somme_nb_vald": [
            df[df["res_com"].str.contains(l, na=False)]["nb_vald"].sum() for l in lignes
        ],
    }
)

lignes_df

In [ ]:
# Validations par ligne et par catégorie de titre (NE PAS UTILISER CAR ERREUR RESULTAT).

# lignes = ['RER A', 'RER B', 'RER C', 'RER D', 'RER E', 'METRO 1', 'METRO 2', 'METRO 3', 'METRO 3bis','METRO 4', 'METRO 5', 'METRO 6', 'METRO 7', 'METRO 7bis','METRO 8', 'METRO 9', 'METRO 10', 'METRO 11', 'METRO 12', 'METRO 13', 'METRO 14', 'TRAIN H', 'TRAIN J', 'TRAIN K', 'TRAIN L', 'TRAIN N', 'TRAIN P', 'TRAIN R', 'TRAIN V', 'TRAM 1', 'TRAM 2', 'TRAM 3', 'TRAM 3a', 'TRAM 3b', 'TRAM 4', 'TRAM 5', 'TRAM 6', 'TRAM 7', 'TRAM 8', 'TRAM 9', 'TRAM 10', 'TRAM 11', 'TRAM 12', 'TRAM 13', 'TRAM 14', 'CABLE 1', 'CDGVAL', 'FUNICULAIRE MONTMARTRE']

# lignes_df = (
#   df[df["res_com"].isin(lignes)]
#   .groupby(["res_com", "categorie_titre"])["nb_vald"]
#   .sum()
#   .reset_index()
# )

# lignes_df

# 4. Export

Exporter les fichiers en format csv.

In [ ]:
# Sauvegarder les données de validations traitées dans un nouveau csv.

processed_validations_filepath = os.path.join(
    "..", "data", "processed", "validations.csv"
)
validations_df.to_csv(processed_validations_filepath, index=False, encoding="utf-8-sig")

print(
    f"Fichier 'données de validations' traité enregistré sous : {processed_validations_filepath}"
)

In [ ]:
# Sauvegarder les données de stations traitées dans un nouveau csv.

processed_stations_filepath = os.path.join("..", "data", "processed", "stations.csv")
stations_df.to_csv(processed_stations_filepath, index=False, encoding="utf-8-sig")

print(
    f"Fichier 'liste des stations' traité enregistré sous : {processed_stations_filepath}"
)

In [ ]:
# Sauvegarder les données fusionnées dans un nouveau csv.

processed_fusion_filepath = os.path.join(
    "..", "data", "processed", "validations_fusion.csv"
)
df.to_csv(processed_fusion_filepath, index=False, encoding="utf-8-sig")

print(
    f"Fichier 'fusion valiations et stations' enregistré sous : {processed_fusion_filepath}"
)

In [ ]:
# Sauvegarder les données de validations par ligne de transport dans un nouveau csv.

processed_ligne_filepath = os.path.join(
    "..", "data", "processed", "validations_ligne.csv"
)
lignes_df.to_csv(processed_ligne_filepath, index=False, encoding="utf-8-sig")

print(f"Fichier 'validations par ligne' enregistré sous : {processed_ligne_filepath}")